# Milestone 0 — Transcript-Check

**Ziel dieser Notebook-Zelle-Serie:** Nur prüfen, ob wir für unser Test-Video überhaupt eine brauchbare Transcript bekommen — kein Chunking, keine Datenbank, nichts Kompliziertes. Das ist reiner Machbarkeits-Check, bevor wir Zeit in den vollen Aufbau stecken.

**Testvideo:** _Nutrients For Brain Health & Performance_ — Huberman Lab Podcast #42
Link: https://www.youtube.com/watch?v=E7W4OQfJWdw


## Schritt 1: Libraries importieren

Wir importieren nur das, was wir für den reinen Check brauchen — die `youtube-transcript-api`, die wir gerade installiert haben.


In [1]:
import sys
print(sys.executable)

/Users/skander/Documents/IronHack/W8/health-fitness-qa-bot/venv/bin/python


In [2]:
# YouTubeTranscriptApi ist die Klasse, die die eigentliche Arbeit macht: Anfrage an YouTube, Untertitel zurückgeben
from youtube_transcript_api import YouTubeTranscriptApi

# Diese Exceptions brauchen wir, um sauber abzufangen, WARUM es fehlschlägt (falls es fehlschlägt),
# statt nur einen generischen Fehler zu sehen
from youtube_transcript_api._errors import TranscriptsDisabled, NoTranscriptFound

## Schritt 2: Video-ID festlegen

Wichtig: Die Library braucht nur die **Video-ID** (den kurzen Code nach `v=` in der URL), nicht die volle URL.


In [3]:
# Video-ID aus https://www.youtube.com/watch?v=E7W4OQfJWdw
video_id = "E7W4OQfJWdw"

print(f"Teste Video-ID: {video_id}")

Teste Video-ID: E7W4OQfJWdw


## Schritt 3: Transcript abrufen (der eigentliche Test)

Wir versuchen, die Transcript zu holen — und fangen mögliche Fehler sauber ab, statt dass das Notebook einfach abstürzt. So sehen wir sofort, ob Plan A (offizielle Untertitel) funktioniert, oder ob wir auf Plan B (Whisper) ausweichen müssen.


In [4]:
try:
    # .fetch() holt die Transcript als Liste von Segmenten (Text + Start-Zeit + Dauer)
    transcript = YouTubeTranscriptApi().fetch(video_id)

    # Erfolg! Kurze Bestätigung, wie viele Segmente wir bekommen haben
    print(f"✅ Transcript gefunden: {len(transcript)} Segmente\n")

    # Zeig die ersten 5 Segmente an, um die Qualität grob einzuschätzen
    for segment in transcript[:5]:
        print(f"[{segment.start:.1f}s] {segment.text}")

except TranscriptsDisabled:
    # Passiert, wenn der Video-Ersteller Untertitel komplett deaktiviert hat
    print("❌ Untertitel sind für dieses Video deaktiviert. Wir brauchen Plan B (Whisper).")

except NoTranscriptFound:
    # Passiert, wenn es zwar Untertitel gibt, aber nicht in einer verfügbaren Sprache
    print("❌ Keine passende Transcript gefunden. Wir brauchen Plan B (Whisper).")

except Exception as e:
    # Auffangnetz für alles andere Unerwartete (z.B. Netzwerkfehler)
    print(f"❌ Unerwarteter Fehler: {e}")

✅ Transcript gefunden: 2502 Segmente

[0.4s] - Welcome to the Huberman Lab Podcast,
[2.3s] where we discuss science
[3.7s] and science-based tools for everyday life.
[6.0s] [upbeat rock music]
[9.4s] I'm Andrew Huberman,


## Interpretation der Ergebnisse

- **Wenn du oben `✅` siehst:** Plan A funktioniert für dieses Video — wir können mit `youtube-transcript-api` weiterarbeiten. Schau dir die 5 Beispiel-Segmente an: Sieht der Text sauber aus (richtige Wörter, sinnvolle Sätze), oder wirkt er fehlerhaft/abgehackt?
- **Wenn du `❌` siehst:** Kein Problem — dann testen wir als Nächstes `yt-dlp` + Whisper als Fallback für genau dieses Video.

**Nächster Schritt (Milestone 1):** Sobald wir wissen, dass Plan A (oder B) funktioniert, machen wir daraus einen sauberen, wiederverwendbaren Codeblock, der für _jedes_ Video-ID funktioniert — nicht nur für unseren Testkandidaten.


## Plan B — Audio-Fallback mit yt-dlp + Whisper (OpenAI API)

**Ziel:** Plan B funktioniert unabhängig davon, ob Plan A für dieses Video nötig war — wir testen ihn jetzt einfach auf demselben Video, um die Code-Logik zu verifizieren. In der echten Nutzung greift Plan B automatisch nur dann, wenn Plan A fehlschlägt.

**Ablauf:** yt-dlp lädt nur die Audiospur herunter (kein Video) → die Audiodatei wird an die Whisper-API von OpenAI geschickt → wir bekommen Text + Zeitstempel zurück.

**Warum die OpenAI-Whisper-API statt einem lokalen Whisper-Modell:** Laut unserer Deployment-Entscheidung bevorzugen wir API-basierte Modelle statt selbst-gehosteter — das lokale Whisper-Modell wäre mehrere GB groß und würde uns beim Deployment genau das Problem bereiten, das wir vermeiden wollen.

**Zwei Voraussetzungen, bevor der Code läuft:**

1. **ffmpeg** muss installiert sein (yt-dlp braucht es, um die Audiospur zu extrahieren). Falls noch nicht vorhanden, im Terminal: `brew install ffmpeg`
2. Eine **`.env`-Datei** im Projektordner mit deinem OpenAI-API-Key (nicht im Notebook selbst, aus Sicherheitsgründen):
   ```
   OPENAI_API_KEY=dein-key-hier
   ```


In [5]:
# Installiert die zwei zusätzlichen Pakete, die wir für Plan B brauchen:
# openai -> um die Whisper-API anzusprechen
# python-dotenv -> um den API-Key sicher aus der .env-Datei zu laden, statt ihn im Code zu schreiben
%pip install openai python-dotenv

  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached anyio-4.14.2-py3-none-any.whl.metadata (4.6 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jiter-0.16.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (5.2 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached tqdm-4.70.0-py3-none-any.whl.metadata (57 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.8.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.46.4-cp313-cp313-macosx_11_0_arm64.whl.metadata (6.6 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 3.5 MB/s  0:00:003.4 MB/s eta 0:00:01
Using cached anyio-4

In [15]:
import os
print(os.getcwd())

/Users/skander/Downloads


In [17]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(dotenv_path="/Users/skander/Documents/IronHack/W8/health-fitness-qa-bot/.env")

client = OpenAI()

print("API-Key gefunden:", os.environ.get("OPENAI_API_KEY") is not None)

API-Key gefunden: True


### Audio herunterladen

**Wichtiger Hinweis:** Wir laden hier absichtlich nur die **ersten 3 Minuten** herunter, nicht die ganze Episode. Grund: Die Whisper-API hat ein Datei-Limit von 25 MB, und für den reinen Funktionstest brauchen wir keine mehrstündige Datei -- das spart außerdem Zeit und API-Kosten.


In [18]:
import yt_dlp

def download_audio(video_id: str, output_dir: str = "audio") -> str:
    url = f"https://www.youtube.com/watch?v={video_id}"
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, f"{video_id}.%(ext)s")

    ydl_opts = {
        "format": "bestaudio/best",           # nur Audio, kein Video -> schneller, kleiner
        "outtmpl": output_path,                # wohin die Datei gespeichert wird
        # nur die ersten 3 Minuten (180 Sekunden) -- reicht für den Test
        "download_ranges": yt_dlp.utils.download_range_func(None, [(0, 180)]),
        "postprocessors": [{
            "key": "FFmpegExtractAudio",
            "preferredcodec": "mp3",
            "preferredquality": "128",
        }],
        "quiet": True,                          # unterdrückt die ausführliche yt-dlp-Konsolenausgabe
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])

    return os.path.join(output_dir, f"{video_id}.mp3")

audio_path = download_audio(video_id)
print(f"Audio gespeichert unter: {audio_path}")

Audio gespeichert unter: audio/E7W4OQfJWdw.mp3         


### Audio an Whisper schicken und Transcript zurückbekommen


In [20]:
def transcribe_with_whisper(audio_path: str):
    with open(audio_path, "rb") as audio_file:
        response = client.audio.transcriptions.create(
            model="whisper-1",
            file=audio_file,
            response_format="verbose_json",        # gibt uns Segmente mit Zeitstempeln zurück, nicht nur reinen Text
            timestamp_granularities=["segment"],
        )
    return response.segments

segments = transcribe_with_whisper(audio_path)

print(f"✅ Whisper-Transcript: {len(segments)} Segmente\n")
for seg in segments[:5]:
    print(f"[{seg.start:.1f}s] {seg.text}")

✅ Whisper-Transcript: 69 Segmente

[0.0s]  Welcome to the Huberman Lab Podcast,
[2.2s]  where we discuss science
[3.7s]  and science-based tools for everyday life.
[9.4s]  I'm Andrew Huberman,
[10.3s]  and I'm a professor of neurobiology and ophthalmology


## Interpretation Plan B

- **Wenn oben `✅` mit Segmenten erscheint:** Plan B funktioniert technisch -- yt-dlp + Whisper-API-Pipeline ist verifiziert und einsatzbereit als Fallback.
- Vergleich mit Plan A (oben): Der Text sollte inhaltlich sehr ähnlich sein (beide beschreiben dieselben ersten 3 Minuten), auch wenn die Segment-Grenzen anders liegen -- Whisper und YouTubes eigene Untertitel teilen Sprache nicht exakt gleich in Segmente.

**Damit sind beide Pfade für Milestone 0 verifiziert.** In Milestone 1/2 bauen wir daraus eine einzige Funktion mit `try/except`-Logik: Plan A zuerst versuchen, bei Fehlschlag automatisch auf Plan B umschalten.


In [21]:
# Neue Video-ID für den Negativ-Test
arabic_video_id = "JM2mF0D5O44"

# Schritt 1: Plan A testen -- wir erwarten hier ein ❌
try:
    transcript = YouTubeTranscriptApi().fetch(arabic_video_id)
    print(f"✅ Plan A hat doch funktioniert: {len(transcript)} Segmente")
except (TranscriptsDisabled, NoTranscriptFound):
    print("❌ Plan A fehlgeschlagen wie erwartet -- keine Untertitel gefunden")
except Exception as e:
    print(f"❌ Anderer Fehler: {e}")

❌ Plan A fehlgeschlagen wie erwartet -- keine Untertitel gefunden


In [22]:
# Schritt 2: Plan B testen -- Audio holen + mit Whisper transkribieren
arabic_audio_path = download_audio(arabic_video_id)
print(f"Audio gespeichert unter: {arabic_audio_path}")

arabic_segments = transcribe_with_whisper(arabic_audio_path)

print(f"✅ Whisper-Transcript: {len(arabic_segments)} Segmente\n")
for seg in arabic_segments[:5]:
    print(f"[{seg.start:.1f}s] {seg.text}")

Audio gespeichert unter: audio/JM2mF0D5O44.mp3          
✅ Whisper-Transcript: 45 Segmente

[0.0s]  كسكسي بالدجاج
[60.0s]  نصفة جلد من الجبن
[62.0s]  نضيف سلطة ماء والخل
[64.0s]  250 غ حمص
[66.0s]  ملح
